In [6]:
import pandas as pd
import json
from pathlib import Path
from pdf2image import convert_from_path
from IPython.display import display, Markdown  # optional
from PIL import Image
import mimetypes

from google import genai
from google.genai import types
from google.genai.types import HttpOptions
from google.oauth2.credentials import Credentials


# --- CONFIGURATION (COMPANY VM) ---

base_url = "https://vertexai.prod.ai-gateway.quantumblack.com/7a4f3d63-b5db-4d5b-8076-8f114d1f14f7/"
access_token = "eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJhZXNKN2kxNGNidnVuTU40MTJrOU5yZ2ROeENhTlJudTNPbC1TU08ycFlJIn0.eyJleHAiOjE3NjQ5MDc0NzQsImlhdCI6MTc2NDkwNTY3NCwiYXV0aF90aW1lIjoxNzY0OTA1Njc0LCJqdGkiOiIzZDJhOTk4Yy0xNTM3LTQ5ZmUtODczYi0zNjU0ZmVkMjQwMjIiLCJpc3MiOiJodHRwczovL2F1dGgubWNraW5zZXkuaWQvYXV0aC9yZWFsbXMvciIsImF1ZCI6ImJjZDIzNzI4LTNkMjctNDQ3Yy1hMGE5LWVhY2FmMzkzYTZmNSIsInN1YiI6IjI0NDRiYzZjLTAwMzctNGIyZS1hYzI3LWZjNTlhNTkxNTM2NiIsInR5cCI6IklEIiwiYXpwIjoiYmNkMjM3MjgtM2QyNy00NDdjLWEwYTktZWFjYWYzOTNhNmY1Iiwic2Vzc2lvbl9zdGF0ZSI6IjM3ZjI2MTMyLTQyN2UtNDQyMi1iYmRmLTAxMDA0YWNmNGU1NiIsImF0X2hhc2giOiItcDdnTUlOZXJhQlQ5WW9LbElDTVNBIiwibmFtZSI6IlVnYW5kaGFyIFZhZGRpIiwiZ2l2ZW5fbmFtZSI6IlVnYW5kaGFyIiwiZmFtaWx5X25hbWUiOiJWYWRkaSIsInByZWZlcnJlZF91c2VybmFtZSI6IjE1ZDhiYmNkMmMzNTNmYWUiLCJlbWFpbCI6IlVnYW5kaGFyX1ZhZGRpQG1ja2luc2V5LmNvbSIsImFjciI6IjEiLCJzaWQiOiIzN2YyNjEzMi00MjdlLTQ0MjItYmJkZi0wMTAwNGFjZjRlNTYiLCJlbWFpbF92ZXJpZmllZCI6dHJ1ZSwiZm1ubyI6IjM0NzI3NiIsImdyb3VwcyI6WyI3YTRmM2Q2My1iNWRiLTRkNWItODA3Ni04ZjExNGQxZjE0ZjciLCJBbGwgRmlybSBVc2VycyJdfQ.g55gAN9vas_qs5pSgpvKStoxQ9CP4nlXll-2Enb-XrRJ9H5Rhw9gG42XV2vsLOJQHW9guxT5clRwwm13pHaXrdwH4oYNDjZmJ3Z9g9Kgh4zZ7-AdbVCAlr1blReRM7F9bjLnysL4ZeXxVTggd5fLZVxoSURAnWGjX-gYlLxD4azCzp1IR3nIYwvxX44qxtTENvZln2hgmpk_Z3XkBrmclawqas_JvV-DJ6ixKhrk91mS_I3DUzm2YFBwuwiedvahf0KpvTyrxU1M3U2gJvpX5XXqH7p41GSQqu0VE5-B4t3liYLIUxuqAvtkVYeN3Al3lnY1zsEPhvQOBYF0yPwVUA"
credentials = Credentials(access_token)

client = genai.Client(
    http_options=HttpOptions(
        api_version="v1",
        base_url=base_url,
    ),
    vertexai=True,
    project="aigateway",
    location="global",
    credentials=credentials,
)

MODEL_NAME = "gemini-2.5-pro"  # or "models/gemini-2.5-pro" if required

print("Legend-based Diagram Symbol Counter Ready (full-image, no tiling).")


# --- STRICT LEGEND-BASED PROMPT (REFINED) ---

def get_universal_symbol_prompt():
    return """
    You are an Engineering Drawing Symbol Counter AI with strong 2D spatial
    understanding.

    INPUT:
    - ONE sheet of a technical drawing (for example, a wiring harness layout).
    - The sheet typically has:
      - A rectangular table titled something like "Legend", "Legend / Symbol",
        "Symbols", or "Key".
      - Each row of that table contains:
          * a small symbol icon (picture)
          * a text label such as "Symbol 1", "Symbol 2", etc.
      - The SAME icons are repeated in the MAIN DRAWING AREA, outside tables,
        connected by wires or lines.

    DEFINITIONS:
    - LEGEND / TABLE REGION:
        Any rectangular tabular area with multiple rows/columns, grid lines,
        and texts arranged in cells, including:
            * the Legend / Symbol table
            * any connector tables
            * any conductor lists
            * any BOM tables
            * any other tabular regions
    - MAIN DRAWING REGION:
        All other parts of the sheet that are NOT inside any table. This is
        where the actual harness/diagram is drawn.

    YOUR JOB:
      1) Find the LEGEND / SYMBOL table (in the table region).
      2) For each row in that LEGEND table, define one symbol.
      3) Then, ONLY in the MAIN DRAWING REGION (outside all tables), count
         how many times each symbol icon appears.

    STEP 1 – LOCATE THE LEGEND / SYMBOL TABLE
    - Find the table whose title/header text indicates legend/symbol/key.
    - Ignore ALL other tables (connector pin tables, conductor lists, BOM, etc.).
    - If you cannot clearly find a legend/symbol/key table:
        Return exactly:
        { "symbols": [] }
        and STOP.

    STEP 2 – EXTRACT SYMBOL DEFINITIONS FROM LEGEND ROWS
    For each legend row:
      - symbol_id:
          The text label in that row.
          Example: "Symbol 1", "Symbol 2", "Switch symbol", etc.
      - description:
          Short description of what it means. If no clear description exists,
          you may set description equal to symbol_id.

    Do NOT create any symbol that is not actually defined as a row in that legend
    table.

    STEP 3 – COUNT ONLY IN MAIN DRAWING REGION (OUTSIDE ALL TABLES)
    For each symbol defined in the legend:

      3.1 Identify the icon for that symbol from the legend row.
      3.2 Using your spatial reasoning, scan ONLY the MAIN DRAWING REGION
          for occurrences of that icon:
          - You MUST ignore any symbol instances that are inside ANY table,
            including:
                * the legend table itself
                * any connector tables
                * any conductor lists
                * any BOM tables
                * any other tabular area with grid lines
          - In other words: DO NOT COUNT icons that are drawn inside
            any table or in any table cell.

      3.3 The count must follow these rules:
          - count = number of distinct and clear occurrences of that icon
            in the MAIN DRAWING REGION (outside tables) on THIS sheet only.
          - If the symbol is defined in the legend but never appears in the
            main drawing region, use count = 0.
          - Prefer under-counting over hallucinating ambiguous matches.
          - Allow simple visual variations:
                * small size changes
                * rotations
                * flips/mirroring
            but only if it is clearly the same icon from the legend.

    VERY IMPORTANT HARD RULE:
    - When you count, you MUST ignore every icon that lies inside any
      table/legend/key region. You are counting ONLY in the free drawing area.

    STEP 4 – OUTPUT FORMAT (STRICT JSON, NO MARKDOWN)
    Return ONLY a single valid JSON object with this shape:

    {
      "symbols": [
        {
          "symbol_id": "Symbol 1",
          "description": "Symbol 1",
          "count": 4
        },
        {
          "symbol_id": "Symbol 2",
          "description": "Symbol 2",
          "count": 2
        }
      ]
    }

    Rules:
      - "symbols" is a JSON array (possibly empty).
      - Each element must have:
          * "symbol_id"   (string or null)
          * "description" (string or null)
          * "count"       (integer, >= 0)
      - "count" must represent ONLY occurrences in the MAIN DRAWING REGION,
        not inside tables.
      - If no legend/symbol/key table exists or you are not confident:
          return exactly:
          { "symbols": [] }
    """


# --- MODEL CALL: ONE IMAGE → SYMBOL JSON ---

def analyze_page_for_symbols(image_path: str):
    """
    Read one page image and ask model to:
      - detect legend/symbol/key table
      - extract symbol list
      - count occurrences ONLY in the main drawing region
        (no counting inside legend/tables)
    """
    print(f"   -> Analyzing: {image_path} ...")

    with open(image_path, "rb") as f:
        image_bytes = f.read()

    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        mime_type = "image/png"

    image_part = types.Part.from_bytes(
        data=image_bytes,
        mime_type=mime_type,
    )

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=[
                get_universal_symbol_prompt(),
                image_part,
            ],
            config=types.GenerateContentConfig(
                response_mime_type="application/json"
            ),
        )

        raw_text = response.text or "{}"

        # Strip code fences if present
        if "```json" in raw_text:
            raw_text = raw_text.split("```json")[1].split("```")[0]
        elif "```" in raw_text:
            raw_text = raw_text.split("```")[1].split("```")[0]

        data = json.loads(raw_text)

    except Exception as e:
        print(f"      [Error] Could not analyze this page: {e}")
        if hasattr(e, "response") and hasattr(e.response, "text"):
            print("      Raw server response:")
            print(e.response.text)
        return {"symbols": []}

    # Normalize structure
    if not isinstance(data, dict):
        data = {"symbols": []}
    if "symbols" not in data or not isinstance(data["symbols"], list):
        data["symbols"] = []

    return data


# --- PIPELINE: FULL IMAGE / PDF (NO TILING) → EXCEL ---

def process_diagram_file(file_path: str):
    """
    - If PDF: convert pages to images (400 dpi).
    - If image: use directly.
    - For each page: get legend-based symbol list + counts.
    - Save Excel:
        Sheet 'Summary'  : total counts per symbol across all pages
        Sheet 'By_Page'  : page-wise counts
    """
    print(f"\n=== Processing Diagram File: {file_path} ===")
    ext = Path(file_path).suffix.lower()
    temp_images = []

    # 1. Convert PDF to images if needed
    if ext == ".pdf":
        print("   -> Converting PDF to images (400 dpi)...")
        pages = convert_from_path(file_path, dpi=400)
        for i, p in enumerate(pages, start=1):
            img_path = f"temp_diagram_page_{i}.png"
            p.save(img_path, "PNG")
            temp_images.append(img_path)
    else:
        temp_images = [file_path]

    total_counts = {}   # key: (symbol_id, description) -> total_count
    page_rows = []      # per-page records

    # 2. Per-page analysis
    for page_index, img in enumerate(temp_images, start=1):
        print(f"\n--- Page {page_index} ---")
        result = analyze_page_for_symbols(img)

        symbols = result.get("symbols", [])
        if not isinstance(symbols, list):
            print("   -> Unexpected 'symbols' format, skipping this page.")
            continue

        for sym in symbols:
            symbol_id   = sym.get("symbol_id")
            description = sym.get("description")
            count       = sym.get("count")

            # if both missing, skip
            if symbol_id is None and description is None:
                continue

            if count is None:
                count = 0

            key = (symbol_id, description)
            total_counts[key] = total_counts.get(key, 0) + count

            page_rows.append({
                "Page": page_index,
                "Symbol_ID": symbol_id,
                "Description": description,
                "Count_On_Page": count,
            })

        if not symbols:
            print("   -> No legend/symbol table detected on this page.")

    # 3. Save Excel
    if total_counts:
        out_name = f"{Path(file_path).stem}_SYMBOL_COUNTS.xlsx"

        summary_rows = []
        for (symbol_id, description), total in total_counts.items():
            summary_rows.append({
                "Symbol_ID": symbol_id,
                "Description": description,
                "Total_Count": total,
            })

        df_summary = pd.DataFrame(summary_rows)
        df_by_page = pd.DataFrame(page_rows)

        with pd.ExcelWriter(out_name, engine="openpyxl") as writer:
            df_summary.to_excel(writer, sheet_name="Summary", index=False)
            df_by_page.to_excel(writer, sheet_name="By_Page", index=False)

        print(f"\n✅ SUCCESS: Legend-based symbol count Excel created -> {out_name}")
    else:
        print("\nNo legend/symbol/key table detected in this file. No Excel created.")

    # 4. Cleanup temp images
    if ext == ".pdf":
        for img in temp_images:
            Path(img).unlink(missing_ok=True)


print("\nLegend-based Symbol Counter (full-image, strict main-area-only) functions loaded.")


Legend-based Diagram Symbol Counter Ready (full-image, no tiling).

Legend-based Symbol Counter (full-image, strict main-area-only) functions loaded.


In [8]:
file_path = "Master test.png"
process_diagram_file(file_path)


=== Processing Diagram File: Master test.png ===

--- Page 1 ---
   -> Analyzing: Master test.png ...

✅ SUCCESS: Legend-based symbol count Excel created -> Master test_SYMBOL_COUNTS.xlsx
